## PS3 - robot strategy battle (Minimax + Alpha-Beta)

MAX robot A and MIN robot B alternate turns collecting energy cells (10 points each), need a depth-limited Minimax and Alpha-Beta version. The original PDF's evaluation function said 'MAX Score - MIN Score + positional advantage' without ever defining positional advantage, the TA clarified it afterward:

Evaluation = (MAX Score - MIN Score) + Positional Advantage
Positional Advantage = Distance(B, nearest remaining E) - Distance(A, nearest remaining E), Manhattan distance, 0 if no energy remains.

In [1]:
import sys
import time

MOVES_NORMAL = [("UP", -1, 0), ("RIGHT", 0, 1), ("DOWN", 1, 0), ("LEFT", 0, -1)]


def read_input():
    data = sys.stdin.read().split("\n")
    idx = 0
    r, c = map(int, data[idx].split())
    idx += 1
    grid = [data[idx + i] for i in range(r)]
    idx += r
    turn = data[idx].strip()
    idx += 1
    depth = int(data[idx].strip())
    idx += 1
    return grid, r, c, turn, depth


def find_positions(grid, r, c):
    a_pos = b_pos = None
    energy = set()
    for i in range(r):
        for j in range(c):
            ch = grid[i][j]
            if ch == "A":
                a_pos = (i, j)
            elif ch == "B":
                b_pos = (i, j)
            elif ch == "E":
                energy.add((i, j))
    return a_pos, b_pos, energy


def in_bounds(r, c, R, C):
    return 0 <= r < R and 0 <= c < C


def manhattan(p1, p2):
    return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])


def nearest_dist(pos, energy):
    if not energy:
        return 0
    return min(manhattan(pos, e) for e in energy)


def evaluate(a_pos, b_pos, energy, max_score, min_score):
    if energy:
        positional_advantage = nearest_dist(b_pos, energy) - nearest_dist(a_pos, energy)
    else:
        positional_advantage = 0
    return (max_score - min_score) + positional_advantage


def get_moves(pos, grid, R, C, energy, use_heuristic_order):
    moves = []
    for name, dr, dc in MOVES_NORMAL:
        nr, nc = pos[0] + dr, pos[1] + dc
        if in_bounds(nr, nc, R, C) and grid[nr][nc] != "#":
            moves.append((name, nr, nc))
    if use_heuristic_order:
        moves.sort(key=lambda m: 0 if (m[1], m[2]) in energy else 1)
    return moves


def search(grid, R, C, a_pos, b_pos, energy, turn, depth, use_ab, use_heuristic_order):
    stats = {"generated": 0, "expanded": 0, "pruned": 0}

    def recurse(a_pos, b_pos, energy, max_score, min_score, turn, depth, alpha, beta):
        stats["expanded"] += 1
        cur_pos = a_pos if turn == "MAX" else b_pos
        moves = get_moves(cur_pos, grid, R, C, energy, use_heuristic_order)

        if depth == 0 or not moves:
            return evaluate(a_pos, b_pos, energy, max_score, min_score), None

        maximizing = turn == "MAX"
        best_val = float("-inf") if maximizing else float("inf")
        best_move = moves[0][0]

        for i, (name, nr, nc) in enumerate(moves):
            stats["generated"] += 1
            new_a, new_b, new_energy = a_pos, b_pos, energy
            new_max, new_min = max_score, min_score

            if turn == "MAX":
                new_a = (nr, nc)
                if (nr, nc) in energy:
                    new_energy = energy - {(nr, nc)}
                    new_max = max_score + 10
            else:
                new_b = (nr, nc)
                if (nr, nc) in energy:
                    new_energy = energy - {(nr, nc)}
                    new_min = min_score + 10

            next_turn = "MIN" if turn == "MAX" else "MAX"
            val, _ = recurse(new_a, new_b, new_energy, new_max, new_min, next_turn, depth - 1, alpha, beta)

            if maximizing and val > best_val:
                best_val, best_move = val, name
            if (not maximizing) and val < best_val:
                best_val, best_move = val, name

            if use_ab:
                if maximizing:
                    alpha = max(alpha, best_val)
                else:
                    beta = min(beta, best_val)
                if alpha >= beta:
                    stats["pruned"] += len(moves) - i - 1
                    break

        return best_val, best_move

    value, move = recurse(a_pos, b_pos, energy, 0, 0, turn, depth, float("-inf"), float("inf"))
    return value, move, stats


def print_result(algo_name, move, value, stats, depth, exec_time, show_pruned):
    print(f"Algorithm: {algo_name}")
    print(f"Best Move: {move}")
    print(f"Evaluation Score = {value}")
    print(f"Nodes Generated = {stats['generated']}")
    print(f"Nodes Expanded = {stats['expanded']}")
    if show_pruned:
        print(f"Nodes Pruned = {stats['pruned']}")
    print(f"Search Depth = {depth}")
    print(f"Execution Time = {exec_time:.6f}")




quick sanity check on manhattan() by itself before trusting it inside eval, two points 3 apart on one axis and 2 on the other should give 5.

In [2]:
print(manhattan((0,0), (3,2)))
print(manhattan((1,1), (1,1)))

5
0


ok that's right (0 for the same point too). now hand-check the whole search on something tiny before the real 5x7 board.

board:
```
A.E
...
B..
```
A at (0,0), B at (2,0), one energy cell at (0,2). depth 2 means MAX moves once, then MIN moves once, then evaluate.

A's only valid moves from a corner are RIGHT -> (0,1) and DOWN -> (1,0) (UP and LEFT walk off the grid).

if A plays RIGHT to (0,1): B's replies are UP -> (1,0) or RIGHT -> (2,1). distance from each B reply to the energy cell (0,2) is 3 either way, and distance from A's (0,1) to the energy is 1, so eval = (0-0) + (3-1) = 2 for both of B's replies, MIN can't do better than 2 here.

if A plays DOWN to (1,0): B's replies land at (1,0) [same cell as A, allowed here since nothing stops robots overlapping] or (2,1). distance from A's (1,0) to the energy is 3 in both branches, and B's distance is also 3, so eval = 0.

MAX compares 2 (via RIGHT) against 0 (via DOWN) and picks RIGHT. so by hand: value 2, move RIGHT.

In [3]:
grid = ['A.E', '...', 'B..']
a_pos, b_pos, energy = find_positions(grid, 3, 3)
value, move, stats = search(grid, 3, 3, a_pos, b_pos, energy, 'MAX', 2, False, False)
print(value, move, stats)

2 RIGHT {'generated': 6, 'expanded': 7, 'pruned': 0}


matches the hand check: value 2, move RIGHT. check Alpha-Beta agrees (it has to, pruning only skips branches that can't change the answer).

In [4]:
ab_value, ab_move, ab_stats = search(grid, 3, 3, a_pos, b_pos, energy, 'MAX', 2, True, False)
print(ab_value, ab_move, ab_stats)

2 RIGHT {'generated': 5, 'expanded': 6, 'pruned': 1}


same value and move, fewer nodes expanded, reports what it pruned. logic checks out, moving to the real board now.

In [5]:
sample1 = '''5 7\nA.E.#..\n.#..#E.\n...#...\n#E#..#.\n....E.B\nMAX\n5'''.split(chr(10))
grid1 = sample1[1:6]
a1, b1, e1 = find_positions(grid1, 5, 7)
print('A at', a1, 'B at', b1, 'energy at', e1)
print('minimax:', search(grid1, 5, 7, a1, b1, e1, 'MAX', 5, False, False)[:2])
print('alpha-beta:', search(grid1, 5, 7, a1, b1, e1, 'MAX', 5, True, False)[:2])

A at (0, 0) B at (4, 6) energy at {(3, 1), (0, 2), (4, 4), (1, 5)}
minimax: (1, 'RIGHT')
alpha-beta: (1, 'RIGHT')


Best Move comes out RIGHT, matches the PDF's example. Evaluation Score comes out 1 though, not the 20 shown in that same example, that number gets printed in the doc before the eval formula is even defined, so treating it as a placeholder and going with the real computed value.

checking the assignment's other sample board too (5x7, depth 4).

In [6]:
sample2 = '''5 7\nA...E..\n.###...\n..E....\n...###.\n.E...B.\nMAX\n4'''.split(chr(10))
grid2 = sample2[1:6]
a2, b2, e2 = find_positions(grid2, 5, 7)
print('minimax:', search(grid2, 5, 7, a2, b2, e2, 'MAX', 4, False, False)[:2])
print('alpha-beta:', search(grid2, 5, 7, a2, b2, e2, 'MAX', 4, True, False)[:2])

minimax: (0, 'RIGHT')
alpha-beta: (0, 'RIGHT')


on both boards, heuristic move order prunes the exact same amount as normal order, which looked off at first. turns out on both boards A's first move from its starting corner is already RIGHT (UP is invalid off the top edge), so the reorder never actually changes anything near the root. built a board on purpose to check the ordering logic isn't just silently broken.

In [7]:
grid3 = ['.....', '.A...', '.E...', '.....', '....B']
a3, b3, e3 = find_positions(grid3, 5, 5)
print('A at', a3, ', its DOWN neighbor is the energy cell, but DOWN is checked 3rd in normal order')
print('normal order   :', search(grid3, 5, 5, a3, b3, e3, 'MAX', 5, True, False))
print('heuristic order:', search(grid3, 5, 5, a3, b3, e3, 'MAX', 5, True, True))

A at (1, 1) , its DOWN neighbor is the energy cell, but DOWN is checked 3rd in normal order
normal order   : (10, 'UP', {'generated': 121, 'expanded': 122, 'pruned': 39})
heuristic order: (10, 'DOWN', {'generated': 96, 'expanded': 97, 'pruned': 47})


there it is, heuristic order expands fewer nodes (97 vs 122) and prunes more (47 vs 39) once the energy-first reorder actually changes which child gets tried first. best move differs between the two (UP vs DOWN) but evaluation score is the same (10), so both are equally good, not a bug.

full script reading stdin and printing in the assignment's format, run for real on the sample input.

In [8]:
import io, sys as _sys
_sys.stdin = io.StringIO('5 7\nA.E.#..\n.#..#E.\n...#...\n#E#..#.\n....E.B\nMAX\n5')
__name__ = '__main__'
import sys
import time

MOVES_NORMAL = [("UP", -1, 0), ("RIGHT", 0, 1), ("DOWN", 1, 0), ("LEFT", 0, -1)]


def read_input():
    data = sys.stdin.read().split("\n")
    idx = 0
    r, c = map(int, data[idx].split())
    idx += 1
    grid = [data[idx + i] for i in range(r)]
    idx += r
    turn = data[idx].strip()
    idx += 1
    depth = int(data[idx].strip())
    idx += 1
    return grid, r, c, turn, depth


def find_positions(grid, r, c):
    a_pos = b_pos = None
    energy = set()
    for i in range(r):
        for j in range(c):
            ch = grid[i][j]
            if ch == "A":
                a_pos = (i, j)
            elif ch == "B":
                b_pos = (i, j)
            elif ch == "E":
                energy.add((i, j))
    return a_pos, b_pos, energy


def in_bounds(r, c, R, C):
    return 0 <= r < R and 0 <= c < C


def manhattan(p1, p2):
    return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])


def nearest_dist(pos, energy):
    if not energy:
        return 0
    return min(manhattan(pos, e) for e in energy)


def evaluate(a_pos, b_pos, energy, max_score, min_score):
    if energy:
        positional_advantage = nearest_dist(b_pos, energy) - nearest_dist(a_pos, energy)
    else:
        positional_advantage = 0
    return (max_score - min_score) + positional_advantage


def get_moves(pos, grid, R, C, energy, use_heuristic_order):
    moves = []
    for name, dr, dc in MOVES_NORMAL:
        nr, nc = pos[0] + dr, pos[1] + dc
        if in_bounds(nr, nc, R, C) and grid[nr][nc] != "#":
            moves.append((name, nr, nc))
    if use_heuristic_order:
        moves.sort(key=lambda m: 0 if (m[1], m[2]) in energy else 1)
    return moves


def search(grid, R, C, a_pos, b_pos, energy, turn, depth, use_ab, use_heuristic_order):
    stats = {"generated": 0, "expanded": 0, "pruned": 0}

    def recurse(a_pos, b_pos, energy, max_score, min_score, turn, depth, alpha, beta):
        stats["expanded"] += 1
        cur_pos = a_pos if turn == "MAX" else b_pos
        moves = get_moves(cur_pos, grid, R, C, energy, use_heuristic_order)

        if depth == 0 or not moves:
            return evaluate(a_pos, b_pos, energy, max_score, min_score), None

        maximizing = turn == "MAX"
        best_val = float("-inf") if maximizing else float("inf")
        best_move = moves[0][0]

        for i, (name, nr, nc) in enumerate(moves):
            stats["generated"] += 1
            new_a, new_b, new_energy = a_pos, b_pos, energy
            new_max, new_min = max_score, min_score

            if turn == "MAX":
                new_a = (nr, nc)
                if (nr, nc) in energy:
                    new_energy = energy - {(nr, nc)}
                    new_max = max_score + 10
            else:
                new_b = (nr, nc)
                if (nr, nc) in energy:
                    new_energy = energy - {(nr, nc)}
                    new_min = min_score + 10

            next_turn = "MIN" if turn == "MAX" else "MAX"
            val, _ = recurse(new_a, new_b, new_energy, new_max, new_min, next_turn, depth - 1, alpha, beta)

            if maximizing and val > best_val:
                best_val, best_move = val, name
            if (not maximizing) and val < best_val:
                best_val, best_move = val, name

            if use_ab:
                if maximizing:
                    alpha = max(alpha, best_val)
                else:
                    beta = min(beta, best_val)
                if alpha >= beta:
                    stats["pruned"] += len(moves) - i - 1
                    break

        return best_val, best_move

    value, move = recurse(a_pos, b_pos, energy, 0, 0, turn, depth, float("-inf"), float("inf"))
    return value, move, stats


def print_result(algo_name, move, value, stats, depth, exec_time, show_pruned):
    print(f"Algorithm: {algo_name}")
    print(f"Best Move: {move}")
    print(f"Evaluation Score = {value}")
    print(f"Nodes Generated = {stats['generated']}")
    print(f"Nodes Expanded = {stats['expanded']}")
    if show_pruned:
        print(f"Nodes Pruned = {stats['pruned']}")
    print(f"Search Depth = {depth}")
    print(f"Execution Time = {exec_time:.6f}")


def main():
    grid, R, C, turn, depth = read_input()
    a_pos, b_pos, energy = find_positions(grid, R, C)

    start = time.time()
    mm_value, mm_move, mm_stats = search(grid, R, C, a_pos, b_pos, energy, turn, depth, False, False)
    mm_time = time.time() - start
    print_result("Minimax", mm_move, mm_value, mm_stats, depth, mm_time, False)

    print()

    start = time.time()
    ab_value, ab_move, ab_stats = search(grid, R, C, a_pos, b_pos, energy, turn, depth, True, False)
    ab_time = time.time() - start
    print_result("Alpha-Beta", ab_move, ab_value, ab_stats, depth, ab_time, True)

    print()

    start = time.time()
    ab2_value, ab2_move, ab2_stats = search(grid, R, C, a_pos, b_pos, energy, turn, depth, True, True)
    ab2_time = time.time() - start
    print_result("Alpha-Beta (heuristic move order)", ab2_move, ab2_value, ab2_stats, depth, ab2_time, True)

    print()
    print("Comparison: Minimax vs Alpha-Beta (normal move order)")
    print(f"Best move -> Minimax = {mm_move}, Alpha-Beta = {ab_move}")
    print(f"Evaluation score -> Minimax = {mm_value}, Alpha-Beta = {ab_value}")
    print(f"Nodes expanded -> Minimax = {mm_stats['expanded']}, Alpha-Beta = {ab_stats['expanded']}")
    print(f"Nodes pruned -> Alpha-Beta = {ab_stats['pruned']}")
    print(f"Execution time -> Minimax = {mm_time:.6f}, Alpha-Beta = {ab_time:.6f}")

    print()
    print("Comparison: Alpha-Beta without move ordering vs with heuristic move ordering")
    print(f"Nodes expanded -> normal order = {ab_stats['expanded']}, heuristic order = {ab2_stats['expanded']}")
    print(f"Nodes pruned -> normal order = {ab_stats['pruned']}, heuristic order = {ab2_stats['pruned']}")
    print(f"Execution time -> normal order = {ab_time:.6f}, heuristic order = {ab2_time:.6f}")


if __name__ == "__main__":
    main()


Algorithm: Minimax
Best Move: RIGHT
Evaluation Score = 1
Nodes Generated = 66
Nodes Expanded = 67
Search Depth = 5
Execution Time = 0.000998

Algorithm: Alpha-Beta
Best Move: RIGHT
Evaluation Score = 1
Nodes Generated = 35
Nodes Expanded = 36
Nodes Pruned = 9
Search Depth = 5
Execution Time = 0.000000

Algorithm: Alpha-Beta (heuristic move order)
Best Move: RIGHT
Evaluation Score = 1
Nodes Generated = 35
Nodes Expanded = 36
Nodes Pruned = 9
Search Depth = 5
Execution Time = 0.000000

Comparison: Minimax vs Alpha-Beta (normal move order)
Best move -> Minimax = RIGHT, Alpha-Beta = RIGHT
Evaluation score -> Minimax = 1, Alpha-Beta = 1
Nodes expanded -> Minimax = 67, Alpha-Beta = 36
Nodes pruned -> Alpha-Beta = 9
Execution time -> Minimax = 0.000998, Alpha-Beta = 0.000000

Comparison: Alpha-Beta without move ordering vs with heuristic move ordering
Nodes expanded -> normal order = 36, heuristic order = 36
Nodes pruned -> normal order = 9, heuristic order = 9
Execution time -> normal order 